In [1]:
import sys
sys.path.append('..')

import torch
from langchain_huggingface import HuggingFaceEmbeddings
import pandas as pd

In [2]:
from src.data.load import load_genres

1. Movies Embedding

In [3]:
genres=load_genres()
genres.head()

,genre,genre_id
0,unknown,0
1,Action,1
2,Adventure,2
3,Animation,3
4,Children's,4


In [4]:
names=genres["genre"].reset_index()
genre_name=names['genre'].to_list()
genre_name


['unknown',
 'Action',
 'Adventure',
 'Animation',
 "Children's",
 'Comedy',
 'Crime',
 'Documentary',
 'Drama',
 'Fantasy',
 'Film-Noir',
 'Horror',
 'Musical',
 'Mystery',
 'Romance',
 'Sci-Fi',
 'Thriller',
 'War',
 'Western']

In [5]:
ids=genres['genre_id'].reset_index()
integers=genres['genre_id'].to_list()

In [6]:
dict_genres=dict(zip(genre_name,integers))

In [7]:
dict_genres

{'unknown': 0,
 'Action': 1,
 'Adventure': 2,
 'Animation': 3,
 "Children's": 4,
 'Comedy': 5,
 'Crime': 6,
 'Documentary': 7,
 'Drama': 8,
 'Fantasy': 9,
 'Film-Noir': 10,
 'Horror': 11,
 'Musical': 12,
 'Mystery': 13,
 'Romance': 14,
 'Sci-Fi': 15,
 'Thriller': 16,
 'War': 17,
 'Western': 18}

In [8]:
from src.data.load import load_items


In [9]:
items=load_items()
items=items.drop(columns=["link","video_release"])
items.tail()

,movie_id,title,year,unknown,Action,Adventure,Animation,Children's,Comedy,Crime,...,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
1677,1678,Mat' i syn (1997),06-Feb-1998,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1678,1679,B. Monkey (1998),06-Feb-1998,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,1,0,0
1679,1680,Sliding Doors (1998),01-Jan-1998,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
1680,1681,You So Crazy (1994),01-Jan-1994,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
1681,1682,Scream of Stone (Schrei aus Stein) (1991),08-Mar-1996,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [10]:

from sklearn.preprocessing import StandardScaler
from sentence_transformers import SentenceTransformer

In [11]:
years=pd.to_numeric(items["year"].str[-4:])
years=years.fillna(years.median()).to_numpy().reshape(-1,1)
years_norm=torch.tensor(StandardScaler().fit_transform(years),dtype=torch.float32)
years_norm.shape


torch.Size([1682, 1])

In [12]:
texts=[]
for _, row in items.iterrows():

    movie_genre=[genre for genre in genre_name if row[genre]==1]
    genres_str=''
    if len(movie_genre[-1:]):
        for genre in movie_genre[:-1]:
            genres_str+=genre+","+" "
    genres_str+=movie_genre[-1]

    texts.append(f"{row['title']} - {genres_str}")
print(texts[:5])

["Toy Story (1995) - Animation, Children's, Comedy", 'GoldenEye (1995) - Action, Adventure, Thriller', 'Four Rooms (1995) - Thriller', 'Get Shorty (1995) - Action, Comedy, Drama', 'Copycat (1995) - Crime, Drama, Thriller']


In [13]:
sentence_transformer = SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\user\movie-recsys\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\user\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (5 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [14]:
enc = sentence_transformer.encode_document(texts,show_progress_bar=False)

In [15]:
encoded_names=torch.tensor(enc, dtype=torch.float32)

In [16]:
emb_movies = torch.cat((years_norm,encoded_names),axis=1)
emb_movies.shape

torch.Size([1682, 385])

In [17]:
torch.save(emb_movies,'movie_features.pt')

2. User Embeddings

In [ ]:
from src.data.load import load_users
from sklearn.preprocessing import StandardScaler
import torch.nn.functional as F

In [21]:
users=load_users()
ids=users["user_id"].to_list()
users=users.drop(columns=["user_id","zip_code"])
users.head()

,age,gender,occupation
0,24,M,technician
1,53,F,other
2,23,M,writer
3,24,M,technician
4,33,F,other


In [27]:
age_tens=torch.tensor((StandardScaler().fit_transform(users[["age"]].to_numpy())).reshape(-1,1),dtype=torch.float32)

In [29]:
gender_tens=torch.tensor((users["gender"]=="M").astype(float).to_numpy().reshape(-1,1),dtype=torch.float32)

In [32]:
job_tens=torch.tensor(pd.get_dummies(users["occupation"].to_list()).to_numpy(),dtype=torch.float32)

In [41]:
user_st_features=torch.cat((gender_tens,job_tens,age_tens),axis=1)

In [43]:
torch.save(user_st_features,"user_static_features.pt")

3. Ratings & history of watching

In [45]:
from src.data.load import load_ratings

In [53]:
ratings = load_ratings()
ratings = ratings[ratings["rating"] >= 4]

ratings["movie_id"] = ratings["movie_id"] - 1
ratings["user_id"] = ratings["user_id"] - 1
ratings =  ratings.sort_values(by='timestamp')

ratings.head()

,user_id,movie_id,rating,timestamp
214,258,254,4,874724710
83965,258,285,4,874724727
43027,258,297,4,874724754
21396,258,184,4,874724781
82655,258,172,4,874724843


In [57]:
train_d,test_d=[],[]

In [59]:
us_films=ratings.groupby("user_id")


for user_id, gr in us_films:
    movies=gr["movie_id"].to_list()
    if len(movies)<3: continue
    split_b=int(len(movies)*0.8)

    for idx in range(1,len(movies)):
        past_movies=movies[max(0,idx-5):idx]
        history_embs=emb_movies[past_movies, 1:].mean(dim=0)

        sample = (user_id, movies[idx], history_embs)
        if idx < split_b:
            train_d.append(sample)
        else:
            test_d.append(sample)


            

